# Week 2 Lab: Responsible AI
**The Evolution of Machine Learning and AI · SETU**

> **How this lab works.** There are **five steps**, each about **20 minutes**. Every step has the same four parts:
> 1. **What you are doing** (plain English, and which lecture slide it matches)
> 2. **Run this** (one cell; you do not change anything)
> 3. **What you should see** (so you know you are on track)
> 4. **Write this down** (the blanks that go into `SUBMISSION.md`)
>
> The whole lab follows **the lecture's pipeline picture**: *World → Data → Labels → Objective → Model → Threshold → Deploy*. Each step is one stop on that pipeline.
>
> The dataset is the file **`data/lending_data.csv`** (8,000 loan applicants). You can open it in Excel; the notebook loads it for you.
> `SUBMISSION.md` is **already filled in** with the reference answers: your job at the end of each step is to **check the numbers on your screen against it** and change any that differ.
>
> **Where the answers come from.** Every answer in `SUBMISSION.md` is read off a printout in this notebook, and every question below says exactly which line or column to look at ("Where to look"). You never need to open the slides, the booklet, the model card or the datasheet to verify an answer. If you find yourself searching other files, stop: the number you want is on screen, above the question.
>
> You do **not** need to read or edit any `.py` file. You do **not** need `shap`, `lime` or `ipywidgets`. You do **not** need an AI assistant: every answer appears on screen after you run the cell.
> **If a step takes more than 40 minutes, stop and post in the channel.** That is a bug in the material, not in you.

**Run the cell below first.** It loads the lab code. (If it errors, run `pip install -r requirements-core.txt` in a terminal, then try again.)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)

import helpers as h          # small wrappers so each step is one cell
import explorer as ex        # the pictures
import mitigate              # the four fixes

lab = h.load_everything()   # reads data/lending_data.csv and trains the standard model
print("Loaded", len(lab["df"]), "applicants from", lab["df"].attrs["source"])
print("You are ready for Step 1.")
lab["df"].head()             # the first five rows of the dataset

---
# Step 1 · Meet the data  *(≈ 20 min · lecture: pipeline stages World → Data → Labels)*

### 1. What you are doing
The lecture said: **the world is the first dataset, and it is not fair.** This lab uses a made-up loan dataset, `data/lending_data.csv`, of 8,000 people in two groups, **A** and **B**. Each person has seven "permitted" facts (income, savings, postcode score, ...) and an outcome column `repaid` (did they pay the loan back?).

Here is the one rule of the dataset, and it is the most important sentence in the lab:

> **Both groups are equally good at repaying loans.** That was written into the data on purpose.

So *any* difference you see between A and B was **manufactured** by one of four mechanisms from the lecture: under-sampling, historical disadvantage, a proxy variable, and biased labels. You will switch them on and off later (optional). For now, just look.

### 2. Run this

In [ ]:
df = lab["df"]
h.describe_mechanisms(df)          # printout A: the four mechanisms and their pipeline stage

print("\nPrintout B: average 'true ability to repay' by group (should be almost identical)")
print(df.groupby("group").latent.mean().round(3).to_string())

print("\nTable C: the population table (the first table of any audit)")
h.population_table(df)[["group", "n", "share", "repaid_rate", "mean_income", "mean_savings",
                        "mean_credit_history_months", "mean_postcode_score"]].round(3)

### 3. What you should see
- **Printout A:** four `[x]` marks, all four mechanisms on, each with its pipeline stage and what it did.
- **Printout B:** two "ability" numbers that are almost the same (about `-0.017` and `-0.005`). **That is the proof the groups are equal.**
- **Table C:** group **B** is about **20%** of the file, has **lower** recorded income and savings, a **shorter** credit history, a postcode score near **0.06** (vs **0.72**), and a recorded repayment rate of about **0.44** (vs **0.55**).

Read that last line again. On paper, B looks riskier. In reality, B is equal. Everything "on paper" was put there by the mechanisms.

### 4. Check these against `SUBMISSION.md`, Step 1
- **1a.** What share of the dataset is group B?
  *Where to look: Table C, row B, column `share`.*
- **1b.** What is the recorded repayment rate for A, and for B?
  *Where to look: Table C, column `repaid_rate`.*
- **1c.** The two groups are *equally* able to repay. So is the difference in 1b caused by **ability** or by **how the data was made**? (one word)
  *Where to look: Printout B shows the two abilities are equal, so the difference cannot be ability. Printout A lists what made it.*
- **1d.** Which lecture pipeline stage does *"some group B people who repaid were recorded as defaulting"* belong to: **World, Data, or Labels**?
  *Where to look: Printout A, the `label_bias` line, column `pipeline stage`. That sentence is printed there word for word.*

**That is Step 1 done.**

---
# Step 2 · Does hiding `group` fix it?  *(≈ 15 min · lecture: "fairness through unawareness")*

### 1. What you are doing
The most common "fix" people suggest: *just don't give the model the `group` column.* The lecture called this **fairness through unawareness** and said it does not work, because other columns **leak** the group.

Test it: train a throw-away model to *guess* someone's group using **only** the seven permitted columns. If it can guess well, the group was never hidden.

### 2. Run this

In [ ]:
from datasets import proxy_recoverability
rec = proxy_recoverability(df)
print(f"Line A: how well can we guess `group` from the permitted columns?  {rec.attrs['accuracy']:.1%}")
print(f"Line B: just guessing 'A' every time would score:                  {rec.attrs['majority_baseline']:.1%}")
print("\nTable C: which column gives the game away (biggest importance = the proxy)")
rec.round(4)

### 3. What you should see
- Guessing accuracy about **99.6%**, against a lazy baseline of about **79.6%**.
- One column with a huge importance (about **0.33**) and the rest near zero. That column is the **proxy**.

### 4. Check these against `SUBMISSION.md`, Step 2
- **2a.** The accuracy of guessing `group` from the permitted columns.
  *Where to look: Line A of the printout.*
- **2b.** The column that leaks the group.
  *Where to look: Table C, top row (the biggest `importance`).*
- **2c.** In one sentence: why does deleting the `group` column *not* stop the model from treating the groups differently?
  *Where to look: 2a and 2b together are the whole answer: the group can be recovered from another column, so it was never hidden.*

**That is Step 2 done.**

---
# Step 3 · The model, and the line you draw  *(≈ 25 min · lecture: pipeline stages Model → Threshold, "where you cut")*

### 1. What you are doing
A standard, well-built model has already been trained for you (it never saw `group`). For each applicant it gives a **score** between 0 and 1: how likely they are to repay. You then choose a **threshold**: approve everyone above the line, refuse everyone below.

The lecture said: **the threshold is one number, and it decides who bears every error.** Here you will look at the usual line (0.5) and then move it.

Four numbers appear per group:
- **approval rate**, share of the group that gets a loan
- **accuracy**, share of decisions that were right
- **TPR**, share of *good* borrowers who were (correctly) approved
- **FPR**, share of *bad* borrowers who were (wrongly) approved

### 2. Run this

In [ ]:
h.try_threshold(lab, 0.5)

### 3. What you should see
At threshold 0.5: group A approved about **60%** of the time, group B about **39%**. The **gap in approval rate is about 0.20**, i.e. twenty percentage points, for two groups that are equally able to repay. Good borrowers in A are approved 78% of the time; in B, 57%.

Now the picture. Two hills (the scores of each group) and one vertical line (the threshold):

In [ ]:
ex.fig_distributions(lab["y"], lab["scores"], lab["group"], 0.5)
plt.show()

Group B's hill sits to the **left**, so the same line cuts off more of B. Nothing about the model is "wrong"; it learned the manufactured data faithfully.

### Move the line
Change the number in the cell below and run it again. Try **0.3**, **0.4**, **0.6**.

In [ ]:
h.try_threshold(lab, 0.4)      # <- change 0.4 and re-run

### The "perfectly fair" line that helps nobody
The cell below scans every threshold and shows the five where the approval-rate gap is **smallest**. Look at the *approval rate overall* column next to it.

In [ ]:
h.find_equal_approval_threshold(lab)

You should see the gap shrink to about **0.02–0.03** only at thresholds like **0.98** (approve **2%** of everyone) or **0.02** (approve **99%** of everyone). The gap is "fair" because almost nobody, or almost everybody, gets a loan. A fairness number quoted **without** the approval rate is meaningless.

### 4. Check these against `SUBMISSION.md`, Step 3
- **3a.** At threshold 0.5: approval rate for A, for B, and the gap.
  *Where to look: the `try_threshold(lab, 0.5)` output. Rows A and B, column `approval rate`; the gap is on the first printed line.*
- **3b.** One other threshold you tried: its gap and its overall approval rate.
  *Where to look: the first printed line of your own `try_threshold` run.*
- **3c.** At the threshold with the smallest gap (0.98), what share of people get a loan?
  *Where to look: the `find_equal_approval_threshold` table, top row, column `approval rate overall`.*
- **3d.** In one sentence: why must a fairness number always be reported **together with** the approval rate?
  *Where to look: 3c is the evidence: a gap of 0.02 sounds fair until you see it approves 2% of people.*

**That is Step 3 done.**

---
# Step 4 · Four fixes, and what each one costs  *(≈ 20 min · lecture: "fairness is not a feature you add")*

### 1. What you are doing
Now try to fix the gap. Four ideas from the lecture, one table:

1. **Drop the proxy column** (`postcode_score`), remove the leak.
2. **Reweight the training data** so group B counts as much as group A.
3. **Different thresholds per group**, tune the line separately for A and B until one gap is zero.
4. **Fix the labels**, go back to pipeline stage *Labels* and use the correct repayment records.

The table shows accuracy and four different gaps. The one to watch first is **`selection_gap`** (the approval-rate gap from Step 3).

### 2. Run this  *(takes about 20 seconds)*

In [ ]:
fixes = mitigate.compare_all(n=8000, seed=1)
fixes.drop(columns=["thresholds"]).round(3)

### 3. What you should see
```
strategy                                    accuracy  selection_gap  tpr_gap  fpr_gap  ppv_gap
baseline (nothing done)                        0.699          0.203    0.205    0.122    0.079
drop postcode_score                            0.684          0.273    0.273    0.202    0.046   <- WORSE
reweighting (pre)                              0.682          0.103    0.115    0.020    0.126
per-group thresholds (post, selection_gap)     0.684          0.000    0.030    0.098    0.163   <- one gap zero, another up
per-group thresholds (post, fpr_gap)           0.689          0.085    0.094    0.001    0.130
fix the label (stage 3)                        0.698          0.033    0.033    0.032    0.009   <- BEST, and no accuracy lost
```
Read it top to bottom:
- Dropping the proxy makes the gap **bigger** (0.27): the model leans harder on the other leaky columns.
- Reweighting helps a bit and costs a bit of accuracy.
- Per-group thresholds zero the gap you tuned for and **push another gap up** (`ppv_gap` 0.08 → 0.16). You cannot have every gap at zero at once; the lecture called this the impossibility result.
- **Fixing the label** removes nearly all of it and costs almost nothing, because that is **where the harm was put in**.

### 4. Check these against `SUBMISSION.md`, Step 4
- **4a.** Which fix made the approval gap **worse**?
  *Where to look: the fixes table, column `selection_gap`: the row where it is bigger than the baseline 0.203.*
- **4b.** Which fix got the gap to exactly 0.000, and which *other* gap went up when it did?
  *Where to look: the row with `selection_gap` 0.000; then read across that row and compare `ppv_gap` with the baseline row.*
- **4c.** Which fix worked best overall, and how much accuracy did it cost?
  *Where to look: the row with the smallest gaps; column `accuracy`, compared with the baseline row.*
- **4d.** In one sentence: why did fixing the *label* work better than fixing the *model*?
  *Where to look: Step 1, Printout A: `label_bias` is a **Labels**-stage mechanism, so the fix that acts at the Labels stage removes it at source.*

**That is Step 4 done.**

---
# Step 5 · The model card and the datasheet  *(≈ 20 min · lecture: pipeline stage Deploy, "who acts on it, and can they refuse it?")*

### 1. What you are doing
Before a model is used on real people, someone writes a **model card** (what the model is for, who it may harm, which fairness trade-off was chosen and why) and a **datasheet** for the data it was trained on. Both are written for you, using the numbers from Steps 1–4:

- `model_card.md`, six boxes.
- `datasheet.md`, what is in `data/lending_data.csv` and how it was made.

### 2. Do this
Open `model_card.md`. Read box 4 (the fairness criterion) and box 5 (the fix and its cost). Compare the numbers in box 5 with your Step 4 table. Then open `datasheet.md` and compare its group counts and repayment rates with your Step 1 table.

### 3. What you should see
Every number in both files matches your run (accuracy 0.698, approval gap 0.033, group B 20.4 %, repayment rates 0.547 / 0.437). If one differs, change it to yours.

### 4. Check these against `SUBMISSION.md`, Step 5
*Where to look: `model_card.md` box 5 and `datasheet.md` "Composition". These two files are only needed here, in Step 5, never for Steps 1 to 4.*

Confirm both files exist and match. Then, at the top of `SUBMISSION.md`, add your student number and name.

**That is the whole lab.** Run `python make_submission.py` and upload the one zip it produces.

---
# Optional: going further  *(not required · only if you want more)*

The original five exercises live here, each cut down to **one cell**.

### A. Which mechanism did the damage?  *(old Exercise 1, now 30 seconds)*
Switches each of the four mechanisms on **alone** and reports the approval-rate gap.

In [ ]:
h.mechanism_table()

You should find `historical_bias` alone produces most of the gap (≈0.15); `label_bias` alone hurts accuracy most; `proxy` alone does nothing *until* another mechanism gives it something to leak. All four together: 0.20.

### B. The impossibility table  *(lecture: "one theorem that says you cannot have everything")*
Each row tunes per-group thresholds to zero **one** gap. Read across the row: the others do not follow.

In [ ]:
fig, table = ex.fig_impossibility(lab["y"], lab["scores"], lab["group"])
plt.show()
table.round(3)

### C. Explain one refused applicant  *(old Exercise 3)*
A from-scratch LIME explanation, no extra installs. Which columns pushed this group-B applicant's score down?

In [ ]:
import explain
from datasets import FEATURES
test, scores, model, train = lab["test"], lab["scores"], lab["model"], lab["train"]
refused = test[(scores < 0.4) & (test.group == "B")].index[0]
local = explain.lime_from_scratch(model, train[FEATURES], test.loc[refused, FEATURES])
explain.fig_local_explanation(local); plt.show()
local.round(4)

An explanation tells you what the model **used**. It does not tell you whether it **should** have. (`postcode_score` barely appears here, yet Step 2 showed it carries the group.)

### D. A simpler model  *(old Exercise 4, the Week 9 preview)*
Is a plain logistic regression fairer than the gradient-boosted model? Compare accuracy **per group**, not just the headline number.

In [ ]:
pd.concat([h.worst_group_accuracy(lab, "gbm"), h.worst_group_accuracy(lab, "logreg")])

### E. The full templates
`model_card_template_full.md` (Mitchell et al.) and `datasheet_template.md` (Gebru et al.) are the professional long forms; the completed short versions in this folder are the format the Week 12 project's responsibility section takes.